# Proyecto 01 · Retail Intelligence — A3: Devoluciones, CRM, Churn e ingreso en riesgo
Tópicos Especiales para la Gestión de Datos (UCAB).

**Qué hace este notebook:** responde las cuatro preguntas de la tarea A3 sobre el esquema `clean` de Supabase:

1. **Devoluciones** — productos, categorías y segmentos con tasas altas, y clientes que devuelven de forma recurrente.
2. **Quejas del CRM** — cambio en la frecuencia de compra antes vs después de una queja no resuelta (con grupo de control de quejas resueltas).
3. **Riesgo de churn** — clientes inactivos según el intervalo mediano entre compras y el valor que concentran.
4. **Ingreso en riesgo a 6 meses** — proyección del gasto reciente de los clientes en riesgo.

**Supuesto metodológico general:** el notebook reporta asociaciones y estimaciones, no causalidad probada.

**Cómo usarlo:** define `DATABASE_URL` (via `.env` en la raíz del repo o como variable de entorno) y ejecuta todo con *Entorno de ejecución → Ejecutar todas*. Requiere `pandas`, `numpy`, `matplotlib`, `sqlalchemy`, `psycopg` y `python-dotenv`.

**Nota de métricas:** las definiciones (ingreso, margen, churn, proyección) siguen el contrato de métricas acordado con el equipo; se marcan las celdas que dependen de una decisión pendiente con Daniel.

## 1. Carga y conexión

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Buscar .env en el directorio actual o en el padre
for ruta in (Path.cwd() / '.env', Path.cwd().parent / '.env'):
    if ruta.exists():
        load_dotenv(ruta)
        break

DATABASE_URL = os.getenv('DATABASE_URL')
if not DATABASE_URL:
    raise SystemExit('Falta DATABASE_URL. Copia .env en la raiz del repo o define la variable de entorno.')

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

transacciones = pd.read_sql('SELECT * FROM clean.transacciones', engine)
devoluciones  = pd.read_sql('SELECT * FROM clean.devoluciones', engine)
clientes      = pd.read_sql('SELECT * FROM clean.clientes', engine)
interacciones = pd.read_sql('SELECT * FROM clean.interacciones_crm', engine)
productos     = pd.read_sql('SELECT * FROM clean.productos', engine)
dim_fecha     = pd.read_sql('SELECT * FROM clean.dim_fecha', engine)

# Convertir columnas de fecha a datetime
for df, cols in [
    (transacciones, ['fecha', 'fecha_devolucion']),
    (devoluciones, ['fecha_devolucion']),
    (clientes, ['fecha_registro', 'primera_compra', 'ultima_compra', 'fecha_alta_efectiva']),
    (interacciones, ['fecha']),
]:
    for c in cols:
        df[c] = pd.to_datetime(df[c], errors='coerce')

print(f'transacciones: {len(transacciones):,} filas')
print(f'devoluciones:  {len(devoluciones):,} filas')
print(f'clientes:      {len(clientes):,} filas')
print(f'interacciones: {len(interacciones):,} filas')
print(f'productos:     {len(productos):,} filas')
print(f'dim_fecha:     {len(dim_fecha):,} filas')

## 2. Parámetros globales

In [ ]:
# Ventana del caso y constantes de análisis
FECHA_CORTE = pd.Timestamp('2024-12-31')   # fin del caso (24 meses)
INICIO = pd.Timestamp('2023-01-01')        # inicio del caso
MESES_HISTORICO = 12                       # ventana para el promedio de gasto reciente
MESES_PROYECCION = 6                       # horizonte de proyeccion de ingreso
FACTORES_SENSIBILIDAD = [0.75, 1.0, 1.5, 2.0]   # multiples del umbral de churn
FACTORES_SENSIBILIDAD_W = [0.5, 1.0, 1.5]       # multiples de la ventana CRM

print(f'Fecha de corte: {FECHA_CORTE.date()}')
print(f'Horizonte de proyeccion: {MESES_PROYECCION} meses')
print(f'Ventana historica para gasto: {MESES_HISTORICO} meses activos')

## 3. Métricas base (funciones reutilizables)

In [ ]:
def pct(parte, total):
    '''Porcentaje seguro (evita division por cero).'''
    return 100.0 * parte / total if total else 0.0

# Definiciones del contrato de metricas (ver README / documento del equipo):
#   ingreso_neto        = monto_neto
#   ingreso_neto_final  = monto_neto - monto_devuelto
#   margen_bruto        = monto_neto - costo_total
#   margen_neto_devol   = 0 si hubo devolucion (supuesto: anula la venta y su margen)
#   tasa_devolucion     = transacciones devueltas / transacciones elegibles
#   ingreso en riesgo   = promedio mensual de los ultimos 12 meses activos x 6

## 4. Sanity check de totales (deben coincidir con A1)

In [ ]:
ingreso_total = transacciones['monto_neto'].sum()
margen_total = transacciones['margen_bruto'].sum()
monto_dev_total = transacciones['monto_devuelto'].sum()

print(f'Ingreso neto total:  {ingreso_total:,.2f}  (esperado 17,392,987.48)')
print(f'Margen bruto total:  {margen_total:,.2f}  (esperado 4,310,329.05)')
print(f'Monto devuelto total: {monto_dev_total:,.2f}')
print(f'Ticket promedio:     {transacciones["monto_neto"].mean():,.2f}')

## 5. Reconciliación de devoluciones (doble fuente)

Fuente canónica: `clean.devoluciones`. Comparamos contra los campos `devuelto` / `monto_devuelto` de `clean.transacciones` para detectar inconsistencias antes de usar los datos.

In [ ]:
ids_tx = set(transacciones['transaccion_id'])
ids_dev = set(devoluciones['transaccion_id'])

# 1) Devoluciones sin transaccion asociada (huerfanas)
solo_dev = devoluciones[~devoluciones['transaccion_id'].isin(ids_tx)]

# 2) Transacciones marcadas devuelto=1 sin fila en devoluciones
flag_sin_dev = transacciones.loc[transacciones['devuelto'] == 1, 'transaccion_id']
flag_sin_dev = flag_sin_dev[~flag_sin_dev.isin(ids_dev)]

# 3) Comparacion de montos por transaccion
dev_agg = devoluciones.groupby('transaccion_id')['monto_devuelto'].sum().rename('monto_dev_tabla').reset_index()
comp = transacciones[['transaccion_id', 'monto_devuelto', 'monto_neto']].merge(
    dev_agg, on='transaccion_id', how='left'
)
comp['monto_dev_tabla'] = comp['monto_dev_tabla'].fillna(0)
comp['dif_monto'] = (comp['monto_dev_tabla'] - comp['monto_devuelto']).round(2)
dif_monto = comp[comp['dif_monto'].abs() > 0.01]

# 4) Devoluciones parciales vs totales
con_dev = comp[comp['monto_dev_tabla'] > 0].copy()
con_dev['es_total'] = con_dev['monto_dev_tabla'] >= con_dev['monto_neto'] - 0.01

print(f'Devoluciones sin transaccion (huerfanas): {len(solo_dev)}')
print(f'Transacciones devuelto=1 sin fila en devoluciones: {len(flag_sin_dev)}')
print(f'Filas con diferencia de monto > 0.01: {len(dif_monto)}')
print(f'Devoluciones totales (monto == neto): {int(con_dev["es_total"].sum())} de {len(con_dev)}')
print(f'Devoluciones parciales (monto < neto): {int((~con_dev["es_total"]).sum())} de {len(con_dev)}')

## 6. Intervalo mediano entre compras

Se usa como ventana de referencia (`W`) para el análisis de CRM y como umbral base de inactividad (churn).

In [ ]:
t = transacciones.sort_values(['cliente_id', 'fecha']).copy()
t['fecha_prev'] = t.groupby('cliente_id')['fecha'].shift(1)
t['dias_entre_compras'] = (t['fecha'] - t['fecha_prev']).dt.days
dias = t['dias_entre_compras'].dropna()

INTERVALO_MEDIANO = float(dias.median())
print(f'Intervalo mediano entre compras (global): {INTERVALO_MEDIANO:.1f} dias')
print(f'Compras consecutivas consideradas: {len(dias):,}')

# Por segmento (contexto; el umbral base sigue siendo el global)
# transacciones ya trae 'segmento' desnormalizado, no hace falta merge
print('\nMediana por segmento (dias):')
print(t.groupby('segmento')['dias_entre_compras'].median().round(1).to_string())

## 7. A3.1 — Devoluciones

In [ ]:
# Tasa global y proporcion de ingreso perdido
n_tx = transacciones['transaccion_id'].nunique()
n_dev = devoluciones['transaccion_id'].nunique()

tx = transacciones.copy()
tx['es_devolucion'] = tx['transaccion_id'].isin(set(devoluciones['transaccion_id']))

elegibles = tx[tx['ventana_devol_incompleta'] == 0]
n_tx_eleg = elegibles['transaccion_id'].nunique()
n_dev_eleg = int(elegibles['es_devolucion'].sum())

tasa_global = pct(n_dev, n_tx)
tasa_elegible = pct(n_dev_eleg, n_tx_eleg)
ingreso_perdido_pct = pct(devoluciones['monto_devuelto'].sum(), transacciones['monto_neto'].sum())

print(f'Transacciones totales: {n_tx:,}')
print(f'Transacciones devueltas: {n_dev:,}')
print(f'Tasa de devolucion (todas): {tasa_global:.2f}%')
print(f'Tasa de devolucion (excluye ventana incompleta): {tasa_elegible:.2f}%')
print(f'Ingreso devuelto / ingreso neto: {ingreso_perdido_pct:.2f}%')

In [ ]:
# Desglose por categoria (tasa y monto)
by_cat = tx.groupby('categoria').agg(
    transacciones=('transaccion_id', 'count'),
    devoluciones=('es_devolucion', 'sum'),
    monto_neto=('monto_neto', 'sum'),
    monto_devuelto=('monto_devuelto', 'sum'),
).reset_index()
by_cat['tasa_%'] = (100 * by_cat['devoluciones'] / by_cat['transacciones']).round(2)
by_cat['%_monto_devuelto'] = (100 * by_cat['monto_devuelto'] / by_cat['monto_devuelto'].sum()).round(2)
by_cat = by_cat.sort_values('tasa_%', ascending=False)

# Categorias anormalmente altas (> 1.5x la mediana de tasa)
mediana_tasa = by_cat['tasa_%'].median()
anomalas = by_cat[by_cat['tasa_%'] > 1.5 * mediana_tasa]

print('Tasa de devolucion por categoria:')
print(by_cat.to_string(index=False))
print(f'\nMediana de tasa por categoria: {mediana_tasa:.2f}%')
print(f'Categorias con tasa > 1.5x mediana ({1.5 * mediana_tasa:.2f}%):')
print(anomalas[['categoria', 'tasa_%']].to_string(index=False))

In [ ]:
# Por segmento
by_seg = tx.groupby('segmento').agg(
    transacciones=('transaccion_id', 'count'),
    devoluciones=('es_devolucion', 'sum'),
    monto_devuelto=('monto_devuelto', 'sum'),
).reset_index()
by_seg['tasa_%'] = (100 * by_seg['devoluciones'] / by_seg['transacciones']).round(2)
by_seg = by_seg.sort_values('tasa_%', ascending=False)

# Top subcategorias y productos por monto devuelto
by_sub = tx.groupby('subcategoria').agg(
    transacciones=('transaccion_id', 'count'),
    devoluciones=('es_devolucion', 'sum'),
    monto_devuelto=('monto_devuelto', 'sum'),
).reset_index()
by_sub['tasa_%'] = (100 * by_sub['devoluciones'] / by_sub['transacciones']).round(2)
by_sub = by_sub.sort_values('monto_devuelto', ascending=False).head(10)

by_prod = tx.groupby('producto_id').agg(
    transacciones=('transaccion_id', 'count'),
    devoluciones=('es_devolucion', 'sum'),
    monto_devuelto=('monto_devuelto', 'sum'),
).reset_index()
by_prod = by_prod.merge(productos[['producto_id', 'nombre', 'categoria']], on='producto_id', how='left')
by_prod = by_prod.sort_values('monto_devuelto', ascending=False).head(10)

# Clientes recurrentes en devoluciones (>= 2)
dev_por_cli = devoluciones.groupby('cliente_id').agg(
    n_devoluciones=('devolucion_id', 'count'),
    monto_devuelto=('monto_devuelto', 'sum'),
).reset_index()
recurrentes = dev_por_cli[dev_por_cli['n_devoluciones'] >= 2].sort_values('n_devoluciones', ascending=False)

print('Tasa de devolucion por segmento:')
print(by_seg.to_string(index=False))
print('\nTop 10 subcategorias por monto devuelto:')
print(by_sub[['subcategoria', 'tasa_%', 'monto_devuelto']].to_string(index=False))
print('\nTop 10 productos por monto devuelto:')
print(by_prod[['producto_id', 'nombre', 'categoria', 'monto_devuelto']].to_string(index=False))
print(f'\nClientes con >=2 devoluciones: {len(recurrentes)} de {dev_por_cli["cliente_id"].nunique()} clientes con devoluciones')
print(f'Devoluciones de recurrentes: {recurrentes["n_devoluciones"].sum()} ({(100 * recurrentes["n_devoluciones"].sum() / len(devoluciones)):.1f}% del total)')

### Figura 1 — Devoluciones por categoría (tasa y monto)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

by_cat_sorted = by_cat.sort_values('tasa_%', ascending=False)
axes[0].bar(by_cat_sorted['categoria'], by_cat_sorted['tasa_%'], color='#d62728')
axes[0].set_title('Tasa de devolución por categoría')
axes[0].set_ylabel('Tasa (%)')
axes[0].tick_params(axis='x', rotation=45)

by_cat_monto = by_cat.sort_values('monto_devuelto', ascending=False)
axes[1].bar(by_cat_monto['categoria'], by_cat_monto['monto_devuelto'], color='#1f77b4')
axes[1].set_title('Monto devuelto por categoría')
axes[1].set_ylabel('Monto devuelto')
axes[1].tick_params(axis='x', rotation=45)

fig.tight_layout()
plt.show()

## 8. A3.2 — CRM: quejas y compras posteriores

In [ ]:
# Recuento de dominios: primero se muestran los valores para decidir
print('Valores de tipo:')
print(interacciones['tipo'].value_counts().to_string())
print('\nValores de resolucion (todas las interacciones):')
print(interacciones['resolucion'].value_counts().to_string())

quejas = interacciones[interacciones['tipo'] == 'queja'].copy()
print('\nValores de resolucion SOLO en quejas:')
print(quejas['resolucion'].value_counts().to_string())

**Celda de decisión (pendiente de confirmar con Daniel).** Según el recuento anterior, define qué valores de `resolucion` cuentan como *no resuelta*. Propuesta conservadora: `sin_resolver` como núcleo; puede sumarse `escalado` (y opcionalmente `parcialmente_resuelto`) si el equipo lo decide.

In [ ]:
# ===== CELDA DE DECISION (ajustar segun acuerdos con el equipo) =====
NO_RESUELTAS = {'sin_resolver'}
# NO_RESUELTAS = {'sin_resolver', 'escalado'}                    # alternativa intermedia
# NO_RESUELTAS = {'sin_resolver', 'escalado', 'parcialmente_resuelto'}  # alternativa amplia
# ==================================================================

quejas_nr = quejas[quejas['resolucion'].isin(NO_RESUELTAS)].copy()
quejas_resueltas = quejas[quejas['resolucion'] == 'resuelto'].copy()
print(f'Quejas no resueltas: {len(quejas_nr)}')
print(f'Quejas resueltas (control): {len(quejas_resueltas)}')

In [ ]:
# Ventanas iguales ancladas a la fecha de cada queja
W = float(INTERVALO_MEDIANO)

# Indice por cliente para velocidad
tx_por_cli = {cli: g['fecha'].to_numpy() for cli, g in transacciones.groupby('cliente_id')}

def frecuencia_antes_despues(quejas_df, dias):
    filas = []
    d = np.timedelta64(int(dias), 'D')
    for cli, fecha in zip(quejas_df['cliente_id'], quejas_df['fecha']):
        fechas = tx_por_cli.get(cli)
        if fechas is None or len(fechas) == 0:
            filas.append((0, 0))
            continue
        f = np.datetime64(fecha)
        lo = f - d
        hi = f + d
        antes = int(((fechas >= lo) & (fechas < f)).sum())
        despues = int(((fechas > f) & (fechas <= hi)).sum())
        filas.append((antes, despues))
    df = quejas_df[['cliente_id', 'fecha']].copy()
    df[['compras_antes', 'compras_despues']] = pd.DataFrame(filas, index=df.index)
    df['delta'] = df['compras_despues'] - df['compras_antes']
    return df

nr = frecuencia_antes_despues(quejas_nr, W)
res_ = frecuencia_antes_despues(quejas_resueltas, W)

print(f'Ventana W = {W:.0f} dias a cada lado de la queja')
print('\nQuejas NO resueltas:')
print(f'  compras promedio antes:   {nr["compras_antes"].mean():.3f}')
print(f'  compras promedio despues: {nr["compras_despues"].mean():.3f}')
print(f'  cambio promedio:          {nr["delta"].mean():.3f}')
print(f'  % con menos compras despues: {100 * (nr["delta"] < 0).mean():.1f}%')
print('\nQuejas resueltas (control):')
print(f'  compras promedio antes:   {res_["compras_antes"].mean():.3f}')
print(f'  compras promedio despues: {res_["compras_despues"].mean():.3f}')
print(f'  cambio promedio:          {res_["delta"].mean():.3f}')

### Figura 2 — Frecuencia de compra antes vs después de la queja

In [ ]:
grupos = ['Quejas no resueltas', 'Quejas resueltas']
antes = [nr['compras_antes'].mean(), res_['compras_antes'].mean()]
despues = [nr['compras_despues'].mean(), res_['compras_despues'].mean()]

x = np.arange(len(grupos))
ancho = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - ancho / 2, antes, ancho, label='Antes de la queja', color='#7f7f7f')
ax.bar(x + ancho / 2, despues, ancho, label='Después de la queja', color='#ff7f0e')
ax.set_xticks(x)
ax.set_xticklabels(grupos)
ax.set_ylabel('Compras promedio en la ventana')
ax.set_title('Frecuencia de compra antes vs después de la queja')
ax.legend()
fig.tight_layout()
plt.show()

### Sensibilidad de la ventana CRM

In [ ]:
sens_rows = []
for f in FACTORES_SENSIBILIDAD_W:
    dias = INTERVALO_MEDIANO * f
    d_nr = frecuencia_antes_despues(quejas_nr, dias)
    d_res = frecuencia_antes_despues(quejas_resueltas, dias)
    sens_rows.append([
        f, round(dias, 1),
        round(d_nr['compras_antes'].mean(), 3),
        round(d_nr['compras_despues'].mean(), 3),
        round(d_nr['delta'].mean(), 3),
        round(d_res['delta'].mean(), 3),
    ])
sens_df = pd.DataFrame(sens_rows, columns=['factor_W', 'dias', 'nr_antes', 'nr_despues', 'nr_delta', 'ctrl_delta'])
print('Sensibilidad del resultado al ancho de ventana:')
print(sens_df.to_string(index=False))

## 9. A3.3 — Riesgo de churn

**Definición:** un cliente está en riesgo si su recencia (`recencia_dias`, calculada respecto al corte 2024-12-31) supera el intervalo mediano entre compras. Se respeta la etiqueta `segmento` tal como viene en `clean.clientes`; los clientes sin compras se reportan aparte.

In [ ]:
con_compras = clientes[clientes['sin_compras'] == 0].copy()
sin_compras = clientes[clientes['sin_compras'] == 1]
UMBRAL_BASE = float(INTERVALO_MEDIANO)

con_compras['en_riesgo'] = con_compras['recencia_dias'] > UMBRAL_BASE
riesgo = con_compras[con_compras['en_riesgo']]

n_riesgo = int(riesgo['cliente_id'].nunique())
monto_riesgo = float(riesgo['monto_total'].sum())
margen_riesgo = float(riesgo['margen_total'].sum())

print(f'Clientes con compras: {len(con_compras)}')
print(f'Clientes sin compras (excluidos): {len(sin_compras)}')
print(f'Umbral de inactividad: recencia > {UMBRAL_BASE:.0f} dias')
print(f'Clientes en riesgo: {n_riesgo} ({100 * n_riesgo / len(con_compras):.1f}% de los que compran)')
print(f'Monto total en riesgo: {monto_riesgo:,.2f}')
print(f'Margen total en riesgo: {margen_riesgo:,.2f}')

In [ ]:
# Valor en riesgo por segmento
seg_riesgo = con_compras.groupby('segmento').agg(
    n_clientes=('cliente_id', 'count'),
    n_en_riesgo=('en_riesgo', 'sum'),
    monto_total=('monto_total', 'sum'),
).reset_index()
monto_en_riesgo = con_compras[con_compras['en_riesgo']].groupby('segmento')['monto_total'].sum().rename('monto_en_riesgo')
seg_riesgo = seg_riesgo.merge(monto_en_riesgo, on='segmento', how='left').fillna(0)
seg_riesgo['%_en_riesgo'] = (100 * seg_riesgo['n_en_riesgo'] / seg_riesgo['n_clientes']).round(1)
seg_riesgo = seg_riesgo.sort_values('monto_en_riesgo', ascending=False)

print('Riesgo de churn por segmento:')
print(seg_riesgo.to_string(index=False))

### Figura 3 — Valor en riesgo por segmento

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
seg_sorted = seg_riesgo.sort_values('monto_en_riesgo', ascending=True)
ax.barh(seg_sorted['segmento'], seg_sorted['monto_en_riesgo'], color='#2ca02c')
ax.set_xlabel('Monto total en riesgo')
ax.set_title('Valor en riesgo de churn por segmento')
fig.tight_layout()
plt.show()

### Sensibilidad del umbral de churn

In [ ]:
sens_churn_rows = []
for f in FACTORES_SENSIBILIDAD:
    u = INTERVALO_MEDIANO * f
    n = int((con_compras['recencia_dias'] > u).sum())
    m = float(con_compras.loc[con_compras['recencia_dias'] > u, 'monto_total'].sum())
    sens_churn_rows.append([f, round(u, 1), n, round(m, 2)])
sens_churn = pd.DataFrame(sens_churn_rows, columns=['factor', 'umbral_dias', 'n_en_riesgo', 'monto_en_riesgo'])
print('Sensibilidad del churn al umbral de inactividad:')
print(sens_churn.to_string(index=False))

## 10. A3.4 — Ingreso en riesgo a 6 meses

Proyección por cliente: **promedio mensual de los últimos 12 meses activos × 6**. Los clientes sin actividad reciente proyectan 0. El resultado es un escenario estimado, no una pérdida comprobada.

In [ ]:
# Gasto mensual por cliente y promedio de los ultimos 12 meses ACTIVOS
tx_mes = transacciones.copy()
tx_mes['mes'] = tx_mes['fecha'].dt.to_period('M')
gasto_mensual = tx_mes.groupby(['cliente_id', 'mes'])['monto_neto'].sum().reset_index()

promedio_12m = (
    gasto_mensual.sort_values(['cliente_id', 'mes'])
    .groupby('cliente_id')['monto_neto']
    .apply(lambda s: float(s.tail(MESES_HISTORICO).mean()))
    .rename('promedio_mensual_12m')
)

clientes_proy = con_compras.merge(promedio_12m.reset_index(), on='cliente_id', how='left')
clientes_proy['promedio_mensual_12m'] = clientes_proy['promedio_mensual_12m'].fillna(0)
clientes_proy['ingreso_proyectado_6m'] = clientes_proy['promedio_mensual_12m'] * MESES_PROYECCION

riesgo_proy = clientes_proy[clientes_proy['en_riesgo']]
ingreso_riesgo_6m = float(riesgo_proy['ingreso_proyectado_6m'].sum())

proy_seg = riesgo_proy.groupby('segmento').agg(
    n_clientes=('cliente_id', 'count'),
    ingreso_proyectado_6m=('ingreso_proyectado_6m', 'sum'),
).reset_index().sort_values('ingreso_proyectado_6m', ascending=False)

print(f'Ingreso proyectado a 6 meses (clientes en riesgo): {ingreso_riesgo_6m:,.2f}')
print('\nProyeccion por segmento:')
print(proy_seg.to_string(index=False))

In [ ]:
# Evitar doble conteo: clientes en riesgo que tambien tienen queja no resuelta
clientes_queja_nr = set(quejas_nr['cliente_id'])
ambos = riesgo_proy[riesgo_proy['cliente_id'].isin(clientes_queja_nr)]

print(f'Clientes en riesgo CON queja no resuelta: {len(ambos)}')
print(f'  (su ingreso proyectado: {ambos["ingreso_proyectado_6m"].sum():,.2f})')
print(f'Clientes en riesgo SIN queja no resuelta: {len(riesgo_proy) - len(ambos)}')
print('Nota: la proyeccion por cliente se cuenta una sola vez; el solape se reporta para evitar sumarlo dos veces en el storytelling.')

## 11. Validaciones finales

In [ ]:
monto_tx = float(transacciones['monto_devuelto'].sum())
monto_dev = float(devoluciones['monto_devuelto'].sum())

print('monto_devuelto en clean.transacciones:', f'{monto_tx:,.2f}')
print('monto_devuelto en clean.devoluciones: ', f'{monto_dev:,.2f}')
print('Diferencia:', f'{monto_tx - monto_dev:,.2f}')

print('\nConteos vs auditoria.metricas_calidad:')
print(f'  clientes:      {len(clientes)} (esperado 8000)')
print(f'  transacciones: {len(transacciones)} (esperado 50000)')
print(f'  devoluciones:  {len(devoluciones)} (esperado 6724)')
print(f'  interacciones: {len(interacciones)} (esperado 4684)')
print(f'  productos:     {len(productos)} (esperado 318)')

## 12. Limitaciones y supuestos

- **No se demuestra causalidad** entre quejas del CRM y caída de compras; se reporta una asociación temporal con grupo de control.
- **La proyección a 6 meses es un escenario estimado** a partir del ritmo de compra reciente, no una pérdida confirmada.
- `margen_neto_devol` de A1 supone que una devolución anula toda la venta y su margen; las devoluciones parciales pueden requerir ajuste.
- `fecha_alta_efectiva` corrige el problema de `fecha_registro`; todos los análisis temporales deben usarla (o `recencia_dias`, ya calculada contra el corte).
- El flag `convertido` de campañas es atribución declarada; no se usa en A3 (corresponde a A4).
- La etiqueta `segmento` se respeta tal como viene en `clean.clientes`; la definición analítica de churn vive aparte.

## 13. Resumen ejecutivo (para A5)

In [ ]:
resumen = pd.DataFrame({
    'Indicador': [
        'Intervalo mediano entre compras (dias)',
        'Tasa de devolucion global (%)',
        'Tasa de devolucion excl. ventana incompleta (%)',
        'Ingreso devuelto / ingreso neto (%)',
        'Clientes con >= 2 devoluciones',
        'Quejas no resueltas analizadas',
        'Delta compras antes/despues (no resueltas)',
        'Delta compras antes/despues (resueltas)',
        'Clientes en riesgo de churn',
        'Monto total en riesgo',
        'Ingreso proyectado a 6 meses (en riesgo)',
    ],
    'Valor': [
        round(INTERVALO_MEDIANO, 1),
        round(tasa_global, 2),
        round(tasa_elegible, 2),
        round(ingreso_perdido_pct, 2),
        int(len(recurrentes)),
        int(len(quejas_nr)),
        round(nr['delta'].mean(), 3),
        round(res_['delta'].mean(), 3),
        n_riesgo,
        round(monto_riesgo, 2),
        round(ingreso_riesgo_6m, 2),
    ],
})
resumen